In [6]:
import numpy as np
from SelfAttention import SelfAttention, Layer_Dense, sigmoid, binary_cross_entropy, binary_cross_entropy_backward

np.random.seed(0)

# -----------------------------
# 1. Tiny sentiment dataset
# -----------------------------

In [7]:
sentences = [
    # positive
    "the movie was good",
    "the film was great",
    "i love this movie",
    "the movie was not bad",
    "the film was very good",
    "the movie was amazing",
    # negative
    "the movie was bad",
    "the film was terrible",
    "i hate this movie",
    "the movie was not good",
    "the film was very bad",
    "the movie was boring"
]

labels = [
    1, 1, 1, 1, 1, 1,   # positive = 1
    0, 0, 0, 0, 0, 0    # negative = 0
]

### Build vocabulary from the sentences

In [8]:
def build_vocab(sentences, extra_tokens=None):
    if extra_tokens is None:
        extra_tokens = ["<pad>", "<unk>"]
    vocab = list(extra_tokens)
    for s in sentences:
        for w in s.strip().split():
            if w not in vocab:
                vocab.append(w)
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    return vocab, word2idx, idx2word

In [9]:
vocab, word2idx, idx2word = build_vocab(sentences)
V = len(vocab)
pad_idx = word2idx["<pad>"]
unk_idx = word2idx["<unk>"]

print("Vocab:", vocab)
print("Vocab size:", V)

Vocab: ['<pad>', '<unk>', 'the', 'movie', 'was', 'good', 'film', 'great', 'i', 'love', 'this', 'not', 'bad', 'very', 'amazing', 'terrible', 'hate', 'boring']
Vocab size: 18


### Convert sentences to sequences of token indices

In [10]:
def encode_sentence(s, word2idx, max_len):
    tokens = s.strip().split()
    ids = [word2idx.get(w, unk_idx) for w in tokens]
    # pad / truncate to max_len
    if len(ids) < max_len:
        ids = ids + [pad_idx] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return np.array(ids, dtype=np.int32)

In [11]:
max_len = max(len(s.split()) for s in sentences)  # no extra pad beyond max sentence length
T = max_len

X_ids = np.stack([encode_sentence(s, word2idx, max_len) for s in sentences])   # (N, T)
y = np.array(labels, dtype=np.float32).reshape(-1, 1)                           # (N, 1)
N = X_ids.shape[0]

### One-hot encode: we want time-major for the attention layer, so per sample we'll do (T, V)

In [12]:
def one_hot(ids, V):
    # ids: (T,)
    x = np.zeros((ids.shape[0], V), dtype=np.float32)
    for t, idx in enumerate(ids):
        x[t, idx] = 1.0
    return x

# -----------------------------
# 2. Model definition
# -----------------------------

In [13]:
d_model = V  # simplest: use one-hot dimension as model dimension
attn = SelfAttention(d_model=d_model)

### We'll pool over time to a single vector, then a dense layer to 1 logit

In [14]:
dense = Layer_Dense(n_inputs=d_model, n_neurons=1)

### Helper: simple mean pooling over time dimension

In [15]:
def mean_pool(sequence):
    """
    sequence: (T, d_model)
    returns: (1, d_model) mean over T
    """
    return np.mean(sequence, axis=0, keepdims=True)  # (1, d_model)

# -----------------------------
# 3. Training loop (SGD)
# -----------------------------

In [16]:
learning_rate = 0.5
n_epochs = 200

In [17]:
for epoch in range(n_epochs):
    epoch_loss = 0.0

    # Shuffle samples each epoch
    indices = np.arange(N)
    np.random.shuffle(indices)

    for idx in indices:
        # ---- Forward pass for one sample ----
        ids = X_ids[idx]                # (T,)
        label = y[idx:idx+1]            # (1, 1)

        # One-hot input, time-major for attention
        x = one_hot(ids, V)             # (T, V)

        # Self-attention
        attn_out, alpha = attn.forward(x)   # (T, d_model), (T, T)

        # Pool across time -> sentence representation
        pooled = mean_pool(attn_out)       # (1, d_model)

        # Dense -> logit
        logit = dense.forward(pooled)      # (1, 1)

        # Sigmoid -> probability
        pred = sigmoid(logit)              # (1, 1)

        # Loss
        loss = binary_cross_entropy(pred, label)
        epoch_loss += loss

        # ---- Backward pass ----
        # dL/d(pred)
        d_pred = binary_cross_entropy_backward(pred, label)   # (1, 1)

        # pred = sigmoid(logit) => dL/dlogit = dL/dpred * pred * (1 - pred)
        d_logit = d_pred * pred * (1.0 - pred)                # (1, 1)

        # Through dense layer
        d_pooled = dense.backward(d_logit)                    # (1, d_model)

        # Through mean pooling:
        # pooled = (1/T) * sum_t attn_out[t]
        # So each time step gets d_attn_out[t] = d_pooled / T
        d_attn_out = np.repeat(d_pooled / T, T, axis=0)       # (T, d_model)

        # Through attention
        d_x = attn.backward(d_attn_out)                       # (T, d_model), ignored for now

        # ---- SGD parameter update ----
        attn.W_q -= learning_rate * attn.dW_q
        attn.W_k -= learning_rate * attn.dW_k
        attn.W_v -= learning_rate * attn.dW_v
        attn.W_o -= learning_rate * attn.dW_o
        dense.weights -= learning_rate * dense.dweights
        dense.biases  -= learning_rate * dense.dbiases

    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{n_epochs} - avg loss: {epoch_loss / N:.4f}")


Epoch 1/200 - avg loss: 0.7548
Epoch 20/200 - avg loss: 0.4856
Epoch 40/200 - avg loss: 0.5112
Epoch 60/200 - avg loss: 0.4617
Epoch 80/200 - avg loss: 0.4462
Epoch 100/200 - avg loss: 0.1846
Epoch 120/200 - avg loss: 0.0020
Epoch 140/200 - avg loss: 0.0008
Epoch 160/200 - avg loss: 0.0005
Epoch 180/200 - avg loss: 0.0003
Epoch 200/200 - avg loss: 0.0002


# -----------------------------
# 4. Inspect attention on examples
# -----------------------------

In [21]:
def predict_and_show_attention(sentence):
    ids = encode_sentence(sentence, word2idx, max_len)  # (T,)
    x = one_hot(ids, V)                                 # (T, V)
    attn_out, alpha = attn.forward(x)                   # (T, d_model), (T, T)
    pooled = mean_pool(attn_out)
    logit = dense.forward(pooled)
    prob = sigmoid(logit)[0, 0]
    pred_label = 1 if prob >= 0.5 else 0

    tokens = sentence.split()
    # For printing, only show first len(tokens) rows/cols of attention
    T_used = len(tokens)
    alpha_used = alpha[:T_used, :T_used]

    print("\nSentence:", sentence)
    print("Tokens:  ", tokens)
    print(f"Predicted sentiment: {'positive' if pred_label == 1 else 'negative'} (p={prob:.3f})")
    print("Attention matrix (truncated):")
    # Pretty-print a small attention matrix
    with np.printoptions(precision=2, suppress=True):
        print(alpha_used)

### Test on seen and slightly modified examples

In [22]:
test_sentences = [
    "the movie was good",
    "the movie was bad",
    "the movie was not good",
    "the movie was not bad",
    "i love this movie",
    "i hate this movie",
]

for s in test_sentences:
    predict_and_show_attention(s)


Sentence: the movie was good
Tokens:   ['the', 'movie', 'was', 'good']
Predicted sentiment: positive (p=0.999)
Attention matrix (truncated):
[[0.13 0.13 0.12 0.03]
 [0.14 0.13 0.12 0.01]
 [0.16 0.15 0.14 0.03]
 [0.18 0.22 0.16 0.03]]

Sentence: the movie was bad
Tokens:   ['the', 'movie', 'was', 'bad']
Predicted sentiment: negative (p=0.001)
Attention matrix (truncated):
[[0.1  0.09 0.09 0.3 ]
 [0.1  0.09 0.08 0.29]
 [0.11 0.1  0.1  0.33]
 [0.23 0.17 0.28 0.15]]

Sentence: the movie was not good
Tokens:   ['the', 'movie', 'was', 'not', 'good']
Predicted sentiment: negative (p=0.000)
Attention matrix (truncated):
[[0.21 0.2  0.19 0.35 0.04]
 [0.25 0.24 0.21 0.28 0.02]
 [0.22 0.21 0.2  0.33 0.04]
 [0.27 0.3  0.21 0.2  0.02]
 [0.27 0.33 0.24 0.12 0.05]]

Sentence: the movie was not bad
Tokens:   ['the', 'movie', 'was', 'not', 'bad']
Predicted sentiment: positive (p=0.999)
Attention matrix (truncated):
[[0.13 0.13 0.12 0.22 0.41]
 [0.14 0.14 0.12 0.16 0.43]
 [0.14 0.13 0.12 0.2  0.41]
 [0